# J08 — Redistribution, évaporation, prélèvement racinaire et bilan hydrique

**Physique et hydrodynamique des sols** — S. J. Gumiere

*Atelier du Jour 8 : drainage interne, évaporation analytique, bilan hydrique saisonnier HYDRUS-1D, irrigation*

> Version **étudiant** : complétez les cellules marquées `# À COMPLÉTER`.

## Mise en place

Les projets HYDRUS-1D de référence sont dans `../../hydrus/` et se lisent avec `hydrus_io.py` (dossier `notebooks/`).
Les données propres à l'atelier sont dans `data/`. Convention HYDRUS : flux positif **vers le haut**, profondeurs négatives, unités cm et jours.

In [ ]:
import sys
sys.path.insert(0, "..")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import optimize, special, integrate
from hydrus_io import read_tlevel, read_obs_node, read_nod_inf, read_balance, read_a_level

plt.rcParams.update({"figure.figsize": (7.5, 4), "axes.grid": True, "grid.alpha": 0.3})
HYD = "../../hydrus"

# sols de Carsel & Parrish (1988) : thr, ths, alpha, n, Ks (cm/j), l
SOLS = {"sable": [0.045, 0.430, 0.145, 2.68, 712.8, 0.5], "loam sableux": [0.065, 0.410, 0.075, 1.89, 106.1, 0.5],
        "loam": [0.078, 0.430, 0.036, 1.56, 24.96, 0.5], "loam limoneux": [0.067, 0.450, 0.020, 1.41, 10.80, 0.5],
        "argile": [0.068, 0.380, 0.008, 1.09, 4.80, 0.5]}

def vg_theta(h, p):
    thr, ths, a, n = p[:4]; m = 1 - 1 / n
    h = np.asarray(h, float)
    Se = np.where(h < 0, (1 + (a * np.abs(h)) ** n) ** (-m), 1.0)
    return thr + (ths - thr) * Se

def vg_K(h, p):
    thr, ths, a, n, Ks, l = p; m = 1 - 1 / n
    h = np.asarray(h, float)
    Se = np.where(h < 0, (1 + (a * np.abs(h)) ** n) ** (-m), 1.0)
    return Ks * Se ** l * (1 - (1 - Se ** (1 / m)) ** m) ** 2

LOAM = SOLS["loam"]

## Exercice 1 — Drainage interne et capacité au champ dynamique  *(≈ 15 min)*

Le projet `J08_redistribution_loam` simule le drainage interne d'un loam de 100 cm initialement à $h = -10$ cm
(surface couverte, flux nul ; drainage libre au bas ; 30 j ; nœuds d'observation à 10, 25, 50 et 90 cm).

1. Lire `OBS_NODE.OUT` (`read_obs_node`) et `T_LEVEL.OUT` (`read_tlevel`). Tracer $\theta(t)$ aux quatre profondeurs et
   $-v_{bot}(t)$ en log-log. Superposer $\theta(h=-100)$ et $\theta(h=-330$ cm$)$ du loam.
2. Ajuster une loi de puissance $q_d = a\,t^{-b}$ sur le flux de drainage et $\theta = a'\,t^{-b'}$ sur $\theta$ à 25 cm
   (`curve_fit`, $t > 1$ j). Interpréter les exposants.
3. Déterminer le temps $t_{cc}$ pour lequel $q_d$ passe sous 0,1 cm/j (1 mm/j), puis $\theta_{cc}$ à 25 cm à cet instant.
   Comparer à $\theta(-100)$ et $\theta(-330)$. Que devient $t_{cc}$ avec un critère de 0,01 cm/j ?
4. Bilan : stock initial et final (`Volume`), drainage cumulé (`sum(vBot)`), fermeture.

In [ ]:
proj = f"{HYD}/J08_redistribution_loam"
tl = read_tlevel(proj)          # index = temps ; colonnes vBot, sum(vBot), Volume, ...
ob = read_obs_node(proj)        # colonnes MultiIndex (noeud, variable) : noeuds 11, 26, 51, 91 = 10, 25, 50, 90 cm
noeuds = {11: 10, 26: 25, 51: 50, 91: 90}

# 1. tracés theta(t) et -vBot(t) (log-log) + seuils theta(-100), theta(-330)
th_100, th_330 = vg_theta(-100, LOAM), vg_theta(-330, LOAM)
# À COMPLÉTER

# 2. lois de puissance (t > 1 j)
f_pow = lambda t, a, b: a * t ** (-b)
# À COMPLÉTER : curve_fit sur -tl["vBot"] et sur ob[(26, "theta")]

# 3. temps t_cc tel que q_d = 0.1 cm/j, theta à 25 cm à t_cc
# À COMPLÉTER

# 4. bilan : Volume initial / final, sum(vBot)
# À COMPLÉTER

## Exercice 2 — Évaporation : solution analytique par transformée de Kirchhoff  *(≈ 15 min)*

Modèle linéarisé (Gardner 1959 ; notes GAE-1004 §3.7) : $K = K_s e^{\alpha_G h}$ et $\theta - \theta_r = (\theta_s - \theta_r)e^{\alpha_G h}$,
donc $D = K_s/[\alpha_G(\theta_s-\theta_r)]$ est constante et $\Phi = \int_{-\infty}^h K\,dh = K/\alpha_G$. Pour un profil semi-infini
initialement à $h_i$ ($\Phi_i = K(h_i)/\alpha_G$) dont la surface sèche brusquement ($\Phi = 0$), le flux d'évaporation vaut

$$e_{sol}(t) = \frac{\Phi_i}{\sqrt{Dt}}\left[\frac{e^{-x^2}}{\sqrt{\pi}} - x\,\mathrm{erfc}(x)\right], \qquad x = \frac{\alpha_G}{2}\sqrt{Dt},$$

et, sans gravité ($\alpha_G \to 0$ dans le crochet), $e_{sol} = \Phi_i/\sqrt{\pi D t} = S_d/(2\sqrt t)$ avec la désorptivité $S_d = 2\Phi_i/\sqrt{\pi D}$.

1. Écrire `e_kirchhoff(t, Ks, alphaG, ths, thr, hi, gravite=True)` et calculer $D$, $\Phi_i$, $S_d$ pour le loam
   ($K_s = 24{,}96$ cm/j, $\alpha_G = 0{,}02$ cm$^{-1}$, $\theta_s = 0{,}43$, $\theta_r = 0{,}078$, $h_i = -50$ cm).
2. Évaporation réelle à deux stades : $e(t) = \min(e_p, e_{sol}(t))$. Pour $e_p$ = 0,3, 0,5 et 0,8 cm/j, tracer $e(t)$ et le cumul $E(t)$
   (`cumulative_trapezoid`) sur 10 j ; déterminer la durée $t_1$ du stade 1 et $E(10$ j$)$.
3. Vérifier le comportement en $t^{-1/2}$ : pente de $\log e_{sol}$ vs $\log t$ sur 0,001–0,01 j puis sur 0,05–0,5 j, avec et sans gravité.
   À partir de quel temps la gravité compte-t-elle ($x = \tfrac{\alpha_G}{2}\sqrt{Dt} \approx 0{,}1$) ?
4. Sensibilité : refaire pour $\alpha_G$ = 0,01 et 0,05 cm$^{-1}$ et pour $h_i$ = $-20$ et $-100$ cm. Commenter.

In [ ]:
def e_kirchhoff(t, Ks, alphaG, ths, thr, hi, gravite=True):
    """Flux d'évaporation (cm/j) du modèle linéarisé, surface à h -> -inf."""
    D = Ks / (alphaG * (ths - thr))
    Phi_i = Ks * np.exp(alphaG * hi) / alphaG
    # À COMPLÉTER : x, puis formule avec / sans gravité
    pass

Ks, aG, ths, thr, hi = 24.96, 0.02, 0.43, 0.078, -50.0
t = np.linspace(1e-4, 10, 5000)

# 2. e = min(ep, e_sol) pour ep = 0.3, 0.5, 0.8 ; cumul E(t) ; t1
# À COMPLÉTER

# 3. pentes log-log (np.polyfit sur log t, log e) sur 0,001-0,01 j et 0,05-0,5 j ; temps tel que x = 0,1
# À COMPLÉTER

# 4. sensibilité alphaG, hi
# À COMPLÉTER

## Exercice 3 — Bilan hydrique d'une saison de maïs (HYDRUS-1D)  *(≈ 20 min)*

Le projet `J08_saison_culture_loam` simule 120 j de maïs sur un loam de 150 cm (condition atmosphérique avec ruissellement,
`hCritA` = 15 000 cm, drainage libre, Feddes maïs, racines linéaires 0–60 cm, $h_0 = -200$ cm). La météo est dans `data/J08_meteo_saison.csv`
(mm/j) : `pluie_mm`, `ET0_mm`, `Kc`, `LAI`, `ETc_mm`, `Ep_mm`, `Tp_mm`.

1. Recalculer $ET_c = K_c\,ET_0$, $E_p = ET_c\,e^{-0{,}463\,LAI}$ et $T_p = ET_c - E_p$ ; vérifier l'accord avec le fichier
   (écart max) et calculer les totaux saisonniers (mm).
2. Lire `T_LEVEL.OUT` : à $t = 120$ j, extraire `sum(Infil)`, `sum(Evap)`, `sum(vRoot)`, `sum(vBot)`, `sum(RunOff)`, `Volume` (initial et final).
   Écrire le bilan $P = I + R$, $\Delta S = I - E_a - T_a - D$ et vérifier la fermeture avec $\Delta$`Volume` ; calculer $E_a/E_p$ et $T_a/T_p$.
   Vérifier `WatBalR` dans `BALANCE.OUT`.
3. Indice de stress journalier : interpoler `sum(vRoot)` à chaque jour entier, en déduire $T_a$ journalier et $T_a/T_p$ ;
   tracer avec la pluie ; lister les périodes où $T_a/T_p < 0{,}5$.
4. Lire `OBS_NODE.OUT` : $\theta(t)$ à 10, 30, 60, 100 cm avec $\theta_{cc} = \theta(-100)$, $\theta_{pf} = \theta(-15\,000)$ et le seuil
   $\theta_{pf} + 0{,}5(\theta_{cc} - \theta_{pf})$. Puis, avec `data/J08_tensiometres.csv` (charges mesurées à 4 dates sur 0–60 cm),
   calculer $T_a/T_p = \omega = \int \alpha(h)\,b(z)\,dz$ par la fonction de Feddes non compensée (P0 = $-15$, POpt = $-30$, P2H = $-325$,
   P2L = $-600$, P3 = $-8000$ cm ; $h_3$ interpolé selon le $T_p$ du jour) avec $b(z)$ linéaire décroissante sur 0–60 cm, chaque tensiomètre
   représentant une tranche de 10 cm. Le projet HYDRUS utilise $\omega_c = 1$ (aucune compensation, bloc G de `SELECTOR.IN`) : comparer au
   rapport instantané `vRoot/rRoot` de `T_LEVEL.OUT` au même instant et discuter les sources d'écart. À titre théorique seulement : que donnerait
   une compensation avec $\omega_c = 0{,}5$ ($T_a/T_p = \min(1, \omega/\omega_c)$, Šimůnek & Hopmans 2009) ?

In [ ]:
met = pd.read_csv("data/J08_meteo_saison.csv")
proj = f"{HYD}/J08_saison_culture_loam"

# 1. partition Ep / Tp
k = 0.463
# À COMPLÉTER : ETc, Ep, Tp recalculés, écart max, totaux

# 2. bilan à 120 j
tl = read_tlevel(proj)
# À COMPLÉTER

# 3. Ta/Tp journalier
jours = np.arange(1, 121)
Ta_cum = np.interp(jours, tl.index, tl["sum(vRoot)"])
# À COMPLÉTER

# 4. theta aux 4 profondeurs + Feddes sur tensiomètres
ob = read_obs_node(proj)     # noeuds 11, 31, 61, 101 = 10, 30, 60, 100 cm
tens = pd.read_csv("data/J08_tensiometres.csv")

def feddes(h, Tp, P0=-15, POpt=-30, P2H=-325, P2L=-600, P3=-8000, r2H=0.5, r2L=0.1):
    # À COMPLÉTER : h3 selon Tp, puis alpha(h) par morceaux
    pass

## Exercice 4 — Calendrier d'irrigation à partir du bilan  *(≈ 15 min)*

Règle de pilotage : on irrigue lorsque le stock $S$ de la zone racinaire (0–60 cm) descend sous $S_{cc} - p\,RU$ avec $p = 0{,}5$,
d'une dose égale au déficit $S_{cc} - S$ (retour à la capacité au champ). $S_{cc} = \theta_{cc} Z_r$, $RU = (\theta_{cc} - \theta_{pf}) Z_r$.

1. Reconstituer le stock 0–60 cm jour par jour à partir des nœuds d'observation : $S = 20\,\theta_{10} + 25\,\theta_{30} + 15\,\theta_{60}$ (cm)
   et tracer avec $S_{cc}$, le seuil et $S_{pf}$. Quel jour le seuil est-il franchi pour la première fois ?
2. Simuler le calendrier : partir de la série HYDRUS et de ses variations journalières $\Delta S_H$ ; une fois irrigué, on suppose que la culture
   transpire à $T_p$ (on retranche en plus le déficit $T_p - T_a$ de la simulation pluviale) et que l'excédent au-dessus de $S_{cc}$ draine.
   Construire la liste des irrigations (jour, dose) et le volume saisonnier (mm et m³/ha).
3. Refaire pour $p = 0{,}3$ et $p = 0{,}7$ ; comparer nombre d'irrigations, doses et volume total. Quelle règle recommander ?

In [ ]:
proj = f"{HYD}/J08_saison_culture_loam"
ob = read_obs_node(proj); tl = read_tlevel(proj)
jours = np.arange(0, 121)
th10, th30, th60 = (np.interp(jours, ob.index, ob[(nd, "theta")]) for nd in (11, 31, 61))
S_H = 20 * th10 + 25 * th30 + 15 * th60          # stock 0-60 cm (cm)
Zr = 60.0
S_cc, S_pf = vg_theta(-100, LOAM) * Zr, vg_theta(-15000, LOAM) * Zr
RU = S_cc - S_pf

# 1. tracé de S_H(t), seuil, premier franchissement
# À COMPLÉTER

# 2. calendrier : boucle jour par jour
Ta_cum = np.interp(jours, tl.index, tl["sum(vRoot)"]); Ta_j = np.diff(Ta_cum)
Tp_j = met["Tp_mm"].to_numpy() / 10
dS_H = np.diff(S_H)
def calendrier(p):
    S = S_H[0]; irrig = []
    for j in range(120):
        # À COMPLÉTER : S = S_H[j+1] tant qu'on n'a pas irrigué, sinon S += dS_H[j] - (Tp_j[j] - Ta_j[j]) plafonné à S_cc ;
        #               test du seuil, dose = S_cc - S, S = S_cc
        pass
    return irrig
# 3. p = 0.3, 0.5, 0.7

## Exercice 5 — Bonus — modèle de bilan à un réservoir  *(≈ facultatif)*

Écrire un modèle journalier à un réservoir (zone racinaire 0–60 cm, réserve utile $RU$) :
$S_{t} = S_{t-1} + P_t + I_{r,t} - E_{a,t} - T_{a,t} - D_t$ avec $T_a = T_p\min(1, S/((1-p)RU))$ (FAO-56, $p = 0{,}5$),
$E_a = E_p\min(1, S/RU)$, $D = \max(0, S - RU)$ ($S$ = eau disponible au-dessus de $\theta_{pf}$).
Initialiser avec $\theta_0 = \theta(-200$ cm$)$. Comparer $T_a$, $E_a$, $D$ cumulés et $T_a/T_p$ journalier à HYDRUS, puis rejouer avec
le calendrier d'irrigation de l'exercice 4.

In [ ]:
def reservoir(P, Ep, Tp, Ir, RU, S0, p=0.5):
    # À COMPLÉTER : boucle journalière, retourne DataFrame (S, Ea, Ta, D)
    pass

## Pour aller plus loin

* Refaire l'exercice 1 avec un profil stratifié (loam sur sable, J7) : comment la barrière capillaire modifie-t-elle $\theta_{cc}$ ?
* Comparer la solution de Kirchhoff (exercice 2) à une simulation HYDRUS-1D d'évaporation (condition atmosphérique, `hCritA` = $10^6$ cm).
* Dans HYDRUS-1D, ajouter les irrigations de l'exercice 4 à `Prec` dans `ATMOSPH.IN` et comparer $T_a/T_p$, drainage et stock.